# 99 — Data Quality Checks

Hard-fail assertions over the Silver and all 9 Gold tables. Notable check:
`decimal_drift_zero` recomputes the total Medicare payment two independent
ways and asserts the difference is under 1¢ across 9.6M rows — the proof
that the DECIMAL(18,2) casts in Silver eliminated floating-point drift.

In [0]:
from pyspark.sql import SparkSession, functions as F

from utils.secrets import configure_adls_oauth
from utils.paths   import SILVER_PHYSICIAN, GOLD

spark = SparkSession.builder.appName("dq_checks").getOrCreate()
configure_adls_oauth(spark, dbutils)

# Sample-mode relaxes the row-count band assertion
dbutils.widgets.dropdown("sample_mode", "false", ["true", "false"])  
SAMPLE_MODE = dbutils.widgets.get("sample_mode") == "true"

silver        = spark.read.format("delta").load(SILVER_PHYSICIAN)
fact          = spark.read.format("delta").load(GOLD("fact_provider_service"))
dim_provider  = spark.read.format("delta").load(GOLD("dim_provider"))
dim_hcpcs     = spark.read.format("delta").load(GOLD("dim_hcpcs"))
dim_geography = spark.read.format("delta").load(GOLD("dim_geography"))

In [0]:
# Decimal-drift probe: two independent ways to compute total Medicare payment.
recomputed_row = silver.agg(
    F.sum(F.col("Avg_Mdcr_Pymt_Amt") * F.col("Tot_Srvcs")).alias("recomputed")
).first()
stored_row = silver.agg(F.sum("Tot_Mdcr_Pymt_Amt").alias("stored")).first()

recomputed = float(recomputed_row["recomputed"] or 0)
stored     = float(stored_row["stored"] or 0)
drift      = abs(recomputed - stored)

In [0]:
fact_count = fact.count()
row_band = (1, 10_000) if SAMPLE_MODE else (100_000, 12_000_000)

checks = [
    ("npi_not_null",
        silver.filter(F.col("Rndrng_NPI").isNull()).count() == 0),
    ("hcpcs_not_null",
        silver.filter(F.col("HCPCS_Cd").isNull()).count() == 0),
    ("pos_in_domain",
        silver.filter(~F.col("Place_Of_Srvc").isin("F", "O", None)).count() == 0),
    ("ent_in_domain",
        silver.filter(~F.col("Rndrng_Prvdr_Ent_Cd").isin("I", "O", None)).count() == 0),
    ("benes_min_11",
        silver.filter(F.col("Tot_Benes") < 11).count() == 0),
    ("ratios_nonneg",
        silver.filter(F.col("Markup_Ratio") < 0).count() == 0),
    ("provider_tier_set",
        silver.filter(F.col("Provider_Tier").isNull()).count() == 0),
    ("fact_row_count_band",
        row_band[0] < fact_count < row_band[1]),
    ("dim_provider_unique",
        dim_provider.count() == dim_provider.select("npi").distinct().count()),
    ("dim_hcpcs_unique",
        dim_hcpcs.count() == dim_hcpcs.select("hcpcs_cd").distinct().count()),
    ("dim_geo_unique",
        dim_geography.count() == dim_geography.select("state_abrvtn").distinct().count()),
    ("dim_geo_reasonable",
        (40 if not SAMPLE_MODE else 1) < dim_geography.count() < 65),
    ("decimal_drift_zero",
        drift < 0.01),
]

failed = [n for n, ok in checks if not ok]
for name, ok in checks:
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

assert not failed, f"DQ failed: {failed}"
print(f"\nAll DQ checks passed. "
      f"Fact rows: {fact_count:,}. Decimal drift across full Silver: ${drift:.4f}")

  PASS  npi_not_null
  PASS  hcpcs_not_null
  PASS  pos_in_domain
  PASS  ent_in_domain
  PASS  benes_min_11
  PASS  ratios_nonneg
  PASS  provider_tier_set
  PASS  fact_row_count_band
  PASS  dim_provider_unique
  PASS  dim_hcpcs_unique
  PASS  dim_geo_unique
  PASS  dim_geo_reasonable
  PASS  decimal_drift_zero

All DQ checks passed. Fact rows: 9,665,252. Decimal drift across full Silver: $0.0025
